**1. Install and Import Libraries**

In [1]:
!pip install opencv-python mediapipe scikit-learn matplotlib tensorflow


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import os
import time
from matplotlib import pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(os.listdir())

['.git', '.ipynb_checkpoints', '0.npy', 'action_model.keras', 'asl_action_model.h5', 'asl_action_model.keras', 'ASL_Data', 'CodeFile.ipynb', 'dataset_prepare.ipynb', 'download_videos.ipynb', 'Extract_Keypoints.ipynb', 'hand_landmarker.task', 'Logs', 'MP_Data', 'pose_landmarker.task', 'selected_videos', 'Untitled3.ipynb', 'videos', 'WLASL_v0.3.json']


**2.Initialize MediaPipe Hand and Pose Landmark Detectors**

In [3]:
# Hand detector
hand_base = python.BaseOptions(model_asset_path="hand_landmarker.task")

hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    num_hands=2
)

hand_detector = vision.HandLandmarker.create_from_options(hand_options)


# Pose detector
pose_base = python.BaseOptions(model_asset_path="pose_landmarker.task")

pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base
)

pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

In [4]:
def mediapipe_detection(frame):

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    hand_results = hand_detector.detect(mp_image)
    pose_results = pose_detector.detect(mp_image)

    return hand_results, pose_results

In [5]:
def draw_styled_landmarks(frame, hand_results, pose_results):

    # HAND CONNECTIONS
    HAND_CONNECTIONS = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (5,9),(9,10),(10,11),(11,12),
        (9,13),(13,14),(14,15),(15,16),
        (13,17),(17,18),(18,19),(19,20),
        (0,17)
    ]

    if hand_results.hand_landmarks:
        for hand in hand_results.hand_landmarks:

            # draw points
            for lm in hand:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x,y), 4, (0,255,0), -1)

            # draw lines
            for connection in HAND_CONNECTIONS:
                start = hand[connection[0]]
                end = hand[connection[1]]

                x1 = int(start.x * frame.shape[1])
                y1 = int(start.y * frame.shape[0])
                x2 = int(end.x * frame.shape[1])
                y2 = int(end.y * frame.shape[0])

                cv2.line(frame, (x1,y1), (x2,y2), (0,255,255), 2)


    # POSE CONNECTIONS
    POSE_CONNECTIONS = [
        (11,13),(13,15),
        (12,14),(14,16),
        (11,12)
    ]

    if pose_results.pose_landmarks:

        pose = pose_results.pose_landmarks[0]

        for lm in pose:
            x = int(lm.x * frame.shape[1])
            y = int(lm.y * frame.shape[0])
            cv2.circle(frame, (x,y), 3, (255,0,0), -1)

        for connection in POSE_CONNECTIONS:
            start = pose[connection[0]]
            end = pose[connection[1]]

            x1 = int(start.x * frame.shape[1])
            y1 = int(start.y * frame.shape[0])
            x2 = int(end.x * frame.shape[1])
            y2 = int(end.y * frame.shape[0])

            cv2.line(frame, (x1,y1), (x2,y2), (255,255,0), 2)

In [6]:
cap = cv2.VideoCapture(0)

In [7]:
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    hand_results, pose_results = mediapipe_detection(frame)

    draw_styled_landmarks(frame, hand_results, pose_results)

    cv2.imshow("ASL Detection Feed", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

**3.Extract Landmark Feature Vectors for Model Training**

In [8]:
def extract_pose(pose_results):

    if pose_results.pose_landmarks:

        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in pose_results.pose_landmarks[0]]
        ).flatten()

    else:
        pose = np.zeros(33*4)

    return pose

In [9]:
def extract_hands(hand_results):

    left = np.zeros(21*3)
    right = np.zeros(21*3)

    if hand_results.hand_landmarks:

        for idx, hand in enumerate(hand_results.hand_landmarks):

            hand_array = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand]
            ).flatten()

            if idx == 0:
                left = hand_array
            elif idx == 1:
                right = hand_array

    return left, right

In [10]:
def extract_keypoints(hand_results, pose_results):

    pose = extract_pose(pose_results)

    left, right = extract_hands(hand_results)

    return np.concatenate([pose, left, right])

In [11]:
keypoints = extract_keypoints(hand_results, pose_results)

print("Feature vector length:", len(keypoints))

Feature vector length: 258


In [12]:
result_test = extract_keypoints(hand_results, pose_results)

In [13]:
33*4 + 21*3 + 21*3

258

In [14]:
np.save('0', result_test)
np.load('0.npy')

array([ 0.34414738,  0.96921599, -1.13161254,  0.99269509,  0.37000638,
        0.91724885, -1.03587329,  0.99220687,  0.38829112,  0.92048621,
       -1.03610492,  0.99178785,  0.4028374 ,  0.92436111, -1.03656328,
        0.99169487,  0.30279583,  0.91089129, -1.07969177,  0.99116838,
        0.27786392,  0.90864325, -1.07980418,  0.99004245,  0.25007769,
        0.90404797, -1.07969189,  0.9889226 ,  0.4058736 ,  0.96364307,
       -0.43428257,  0.9905026 ,  0.21590638,  0.93244088, -0.64374244,
        0.98720205,  0.37091887,  1.04526532, -0.88788998,  0.94048566,
        0.28964111,  1.03842807, -0.95033526,  0.9373641 ,  0.43780187,
        1.29051948, -0.13483787,  0.57081431,  0.0709945 ,  1.25654078,
       -0.41406232,  0.5015316 ,  0.44715524,  1.50757563, -0.12248614,
        0.2258075 ,  0.01180935,  1.49984157, -0.82194948,  0.24005416,
        0.45559952,  1.36090922, -0.27654314,  0.15566255,  0.04334831,
        1.37316751, -1.28887546,  0.1284811 ,  0.46575683,  1.33

**4.Create Dataset Directory Structure for Training Data**

In [15]:
import os

In [16]:
DATA_PATH = os.path.join("ASL_Data")

In [17]:
no_sequences = 30
sequence_length = 30

In [18]:
actions = np.array([f"sign_{i}" for i in range(60)])

In [30]:
import os

VIDEO_PATH = "selected_videos"

video_files = sorted(os.listdir(VIDEO_PATH))
print(video_files)

['1466681314.2550.mp4', '1466684225.687.mp4', '1467773819.9825.mp4', '1468549286.5649.mp4', '1468580671.3652.mp4', '1468755772.9751.mp4', '1471136062.8464.mp4', '1522767129.7038.mp4', '153289.mp4', '1539176590.1089.mp4', '1546575308.9841.mp4', '1_jXxjd0lKE.mp4', '47501.mp4', '50888.mp4', '537MWOtCl78.mp4', '5IGeT5NK79A.mp4', 'BqmquW0f1ok.mp4', 'CANDY_1-67.mp4', 'CANDY_2-1014.mp4', 'ChIupJBcxjs.mp4', 'Computer 3-ZIm7kG5ie_M.mp4', 'DEAF-103.mp4', 'DRINK-119.mp4', 'EfJ4xLiX5IA.mp4', 'IYH-gBXXl8I.mp4', 'Kwvw-K6GYW8.mp4', 'LMQ4kaHPS48.mp4', 'NO-512.mp4', 'Nc7rSopCpI8.mp4', 'OHHKerxHJjQ.mp4', 'PPmQd2zWdP0.mp4', 'Qvn5RGyb0po.mp4', 'SiBub8VD5wo.mp4', "SignSchool Candy, It's Nothing-YX3NQ7iEwiE.mp4", 'SignSchool Computer 1-_2YnCKHJy6U.mp4', 'SignSchool Computer 4-1f6Cc5NXBgw.mp4', 'SignSchool Deaf 2-1tMYCoZB0lE.mp4', 'SignSchool Deaf 2-VoHLEG8FIH4.mp4', 'SignSchool Last-0SLfkAHELAs.mp4', 'SignSchool Republic of Congo-b6U1CIWIPIw.mp4', 'SignSchool Who-rehp_kwiS-I.mp4', 'SignSchool Who-wmIY6Yo-vp

In [19]:
for action in actions:
    for sequence in range(no_sequences):
        dir_path = os.path.join(DATA_PATH, action, str(sequence))
        os.makedirs(dir_path, exist_ok=True)

**5.Collect Training Sequences and Save Landmark Keypoints**

In [22]:
cap = cv2.VideoCapture(0)

In [23]:
stop_collection = False 

In [24]:
for action in actions:
    for sequence in range(no_sequences):
        for frame_num in range(sequence_length):

            ret, frame = cap.read()
            if not ret:
                stop_collection = True
                break

            hand_results, pose_results = mediapipe_detection(frame) # Run MediaPipe detection
            draw_styled_landmarks(frame, hand_results, pose_results) # Draw landmarks
            keypoints = extract_keypoints(hand_results, pose_results) # Extract keypoints

            npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num)) # Save keypoints
            np.save(npy_path, keypoints)

            # Display status on screen
            cv2.putText(
                frame,
                f"Collecting {action} | Video {sequence}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.imshow("ASL Data Collection", frame)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                stop_collection = True
                break

        if stop_collection:
            break
    if stop_collection:
        break

In [25]:
cap.release()
cv2.destroyAllWindows()

**6.Dataset Assembly & Train/Test Split**

In [20]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [21]:
label_map = {label: num for num, label in enumerate(actions)}

In [22]:
actions = np.array(sorted(os.listdir(DATA_PATH)))

In [24]:
label_map = {label: num for num, label in enumerate(actions)}
print(label_map)

{'1466681314.2550.mp4': 0, '1466684225.687.mp4': 1, '1467773819.9825.mp4': 2, '1468549286.5649.mp4': 3, '1468580671.3652.mp4': 4, '1468755772.9751.mp4': 5, '1471136062.8464.mp4': 6, '1522767129.7038.mp4': 7, '153289.mp4': 8, '1539176590.1089.mp4': 9, '1546575308.9841.mp4': 10, '1': 11, '47501.mp4': 12, '50888.mp4': 13, '537mwotcl78.mp4': 14, '5iget5nk79a.mp4': 15, 'bqmquw0f1ok.mp4': 16, 'candy': 18, 'chiupjbcxjs.mp4': 19, 'computer 3-zim7kg5ie': 20, 'deaf-103.mp4': 21, 'drink-119.mp4': 22, 'efj4xlix5ia.mp4': 23, 'iyh-gbxxl8i.mp4': 24, 'kwvw-k6gyw8.mp4': 25, 'lmq4kahps48.mp4': 26, 'no-512.mp4': 27, 'nc7rsopcpi8.mp4': 28, 'ohhkerxhjjq.mp4': 29, 'ppmqd2zwdp0.mp4': 30, 'qvn5rgyb0po.mp4': 31, 'sibub8vd5wo.mp4': 32, "signschool candy, it's nothing-yx3nq7iewie.mp4": 33, 'signschool computer 1-': 34, 'signschool computer 4-1f6cc5nxbgw.mp4': 35, 'signschool deaf 2-1tmycozb0le.mp4': 36, 'signschool deaf 2-vohleg8fih4.mp4': 37, 'signschool last-0slfkahelas.mp4': 38, 'signschool republic of congo-

In [30]:
sequences = []
labels = []

for action in actions:
    for sequence in range(no_sequences):

        window = []
        valid_sequence = True

        for frame_num in range(sequence_length):

            file_path = os.path.join(
                DATA_PATH, action, str(sequence), f"{frame_num}.npy"
            )

            if os.path.exists(file_path):
                res = np.load(file_path)
            else:
                res=np.zeros(258)

            window.append(res)

        if valid_sequence:
            sequences.append(window)
            labels.append(label_map[action])

In [31]:
X = np.array(sequences)
y = to_categorical(labels, num_classes=len(actions)).astype(int)

print("X shape:",X.shape)
print("y shape:",y.shape)

X shape: (1800, 30, 258)
y shape: (1800, 60)


In [32]:
print("Sequences collected:", len(sequences))
print("Labels collected:", len(labels))

Sequences collected: 1800
Labels collected: 1800


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (1440, 30, 258)
Test: (360, 30, 258)


In [34]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(1440, 30, 258)
(1440, 60)
(360, 30, 258)
(360, 60)


**7.Build and train LSTM Neural Network**

In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import os
import numpy as np

In [39]:
actions = np.array(actions)
log_dir = os.path.join("Logs")
tb_callback = TensorBoard(log_dir=log_dir)

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [40]:
model = Sequential()

model.add(Input(shape=(30, X.shape[2])))

model.add(LSTM(64, return_sequences=True))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(64))

model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))

model.add(Dense(len(actions), activation='softmax'))

In [41]:
model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [42]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                        │ (None, 30, 64)              │          82,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 30, 128)             │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 60)                  │           1,980 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 239,132 (934.11 KB)

 Trainable params: 239,132 (934.11 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
pip install tensorboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
     --- ------------------------------------ 0.5/5.5 MB 15.7 MB/s eta 0:00:01
     -------- ------------------------------- 1.1/5.5 MB 11.9 MB/s eta 0:00:01
     ------------ --------------------------- 1.8/5.5 MB 13.9 MB/s eta 0:00:01
     ----------------- ---------------------- 2.4/5.5 MB 12.9 MB/s eta 0:00:01
     --------------------- ------------------ 3.0/5.5 MB 12.8 MB/s eta 0:00:01
     ------------------------- -------------- 3.6/5.5 MB 12.6 MB/s eta 0:00:01
     ----------------------------- ---------- 4.1/5.5 MB 12.5 MB/s eta 0:00:01
     -------------------------------- ------- 4.5/5.5 MB 12.0 MB/s eta 0:00:01
     ----------------------------------- ---- 4.9/5.5 MB 11.7 MB/s eta 0:00:01
     ---------------------------------------  5.4/5.5 MB 11.9 MB/s eta 0:00:01
     ---------------------------------------  5.5/5.5 MB 11.4 MB/s eta 0:00:01
     ---------------------------------------- 5.5/5.5 MB 10.

In [44]:
model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test),
    callbacks=[tb_callback, early_stop]
)

Epoch 1/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 14s 64ms/step - accuracy: 0.0257 - loss: 4.0943 - val_accuracy: 0.0333 - val_loss: 4.0924
Epoch 2/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.0333 - loss: 4.0923 - val_accuracy: 0.0333 - val_loss: 4.0907
Epoch 3/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.0333 - loss: 4.0907 - val_accuracy: 0.0333 - val_loss: 4.0891
Epoch 4/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.0333 - loss: 4.0892 - val_accuracy: 0.0333 - val_loss: 4.0877
Epoch 5/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.0333 - loss: 4.0878 - val_accuracy: 0.0333 - val_loss: 4.0864
Epoch 6/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.0333 - loss: 4.0866 - val_accuracy: 0.0333 - val_loss: 4.0853
Epoch 7/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.0333 - loss: 4.0855 - val_accuracy: 0.0333 - val_loss: 4.0842
Epoch 8/30
90/90 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.0333 - loss: 4.0846 - val_accuracy: 0.0333 - 

In [45]:
model.save("asl_action_model.keras")

**8.Make Predictions**

In [157]:
res = model.predict(X_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 799ms/step


In [158]:
actions[np.argmax(res[4])]

np.str_('iloveyou')

In [159]:
actions[np.argmax(y_test[4])]

np.str_('yes')

**9.Save Weights**

In [160]:
model.save("asl_action_model.keras")

In [161]:
from tensorflow.keras.models import load_model

In [162]:
model = load_model("asl_action_model.keras")

**10.Evaluation using Confusion Matrix and Accuracy**

In [163]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [164]:
y_pred = model.predict(X_train)

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 235ms/step


In [165]:
y_true = np.argmax(y_train, axis=1)
y_pred = np.argmax(y_pred, axis=1)

In [166]:
multilabel_confusion_matrix(y_true, y_pred)

array([[[116,   0],
        [  0,  24]],

       [[116,   0],
        [  0,  24]],

       [[114,   2],
        [  5,  19]],

       [[111,   5],
        [  2,  22]],

       [[116,   0],
        [  0,  24]],

       [[120,   0],
        [  0,  20]]])

In [167]:
y_pred = model.predict(X_test)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred, axis=1)

multilabel_confusion_matrix(y_true, y_pred)
accuracy_score(y_true, y_pred)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step


0.8857142857142857

**11.Real_Time Testing**

In [46]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import os

In [47]:
DATA_PATH = "ASL_Data"
actions = np.array(sorted(os.listdir(DATA_PATH)))

In [48]:
model = load_model("asl_action_model.keras")

In [175]:
cap = cv2.VideoCapture(0)

In [176]:
sequence = []
sentence = []
predictions=[]
threshold = 0.8

In [ ]:
while True:

    ret, frame = cap.read()
    if not ret:
        print("Frame not captured")
        break

    # MediaPipe detection
    hand_results, pose_results = mediapipe_detection(frame)

    # Draw landmarks
    draw_styled_landmarks(frame, hand_results, pose_results)

    # Extract keypoints
    keypoints = extract_keypoints(hand_results, pose_results)

    sequence.append(keypoints)
    sequence = sequence[-30:]

    threshold=0.9

    # Prediction
    if len(sequence) == 30:

        res = model.predict(np.expand_dims(sequence, axis=0))[0]

        predictions.append(np.argmax(res))
        predictions = predictions[-10:]

        if len(predictions) > 0 and np.bincount(predictions).argmax() == np.argmax(res):

            if res[np.argmax(res)] > threshold:

                if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                    sentence.append(actions[np.argmax(res)])

        if len(sentence) > 5:
            sentence = sentence[-5:]

    # Display prediction
    cv2.putText(
        frame,
        ' '.join(sentence),
        (20,450),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (255,255,255),
        2,
        cv2.LINE_AA
    )

    cv2.imshow("ASL Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━

In [ ]:
cap.release()
cv2.destroyAllWindows()

In [ ]:
res = model.predict(X_test)

for i in range(10):
    print("Predicted:", actions[np.argmax(res[i])],
          " | True:", actions[np.argmax(y_test[i])])